In [2]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [3]:
result_path = Path("../results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", "fw-mrs-temperature",  "fw-mrs-temperature-svm",   
"soft-mrs-linear", "soft-mrs-exponential"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class", "less_negative_class"]
metrics = ["AUROC", "AUPRC"]
less_bias_strengths = ["0.1", "0.2", "0.3"]
mean_bias_strengthts = ["0.8", "0.9"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction"]

In [4]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                result_file = pd.read_json(str(json_file))
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [5]:
result_df = result_df.replace({"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                               "soft-mrs-linear": "Soft-MRS-Linear", "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "fw-mrs-temperature-svm": "FW-MRS-SVM", "fw-mrs-temperature": "FW-MRS-RF"})
result_df


,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.871233,0.009929,0.828050,0.016532,less_positive_class,0.1,0.00,0.000000
1,Uniform,folktables_employment,0.876815,0.009341,0.835082,0.014809,less_positive_class,0.2,0.00,0.000000
2,Uniform,folktables_employment,0.879325,0.008763,0.839663,0.014434,less_positive_class,0.3,0.00,0.000000
3,PSA,folktables_employment,0.867536,0.011096,0.824467,0.017054,less_positive_class,0.1,0.04,0.280000
4,PSA,folktables_employment,0.874352,0.009985,0.833065,0.016079,less_positive_class,0.2,0.10,0.412311
...,...,...,...,...,...,...,...,...,...,...
235,Soft-MRS-Linear,loan_prediction,0.731775,0.051293,0.829830,0.038869,less_negative_class,0.2,0.00,0.000000
236,Soft-MRS-Linear,loan_prediction,0.744113,0.047508,0.842307,0.035076,less_negative_class,0.3,0.00,0.000000
237,Soft-MRS-Exponential,loan_prediction,0.697400,0.070200,0.811018,0.045018,less_negative_class,0.1,0.00,0.000000
238,Soft-MRS-Exponential,loan_prediction,0.728910,0.053409,0.828640,0.040672,less_negative_class,0.2,0.00,0.000000


In [6]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset in datasets:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 3))

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.871\pm0.01$ & $0.838\pm0.014$ & $0.754\pm0.019$ & $0.988\pm0.006$ & $0.67\pm0.079$ & & \\
	& PSA & $0.868\pm0.011$ & $0.83\pm0.014$ & $0.753\pm0.02$ & $0.988\pm0.006$ & $0.642\pm0.094$ & & \\
	& KMM & $0.856\pm0.014$ & $0.819\pm0.014$ & $0.753\pm0.018$ & $0.99\pm0.005$ & $0.615\pm0.087$ & & \\
	& MRS & $0.867\pm0.011$ & $0.836\pm0.013$ & $0.754\pm0.018$ & $0.99\pm0.005$ & $0.648\pm0.077$ & & \\
	& FW-MRS-RF & $0.861\pm0.011$ & $0.832\pm0.014$ & $0.754\pm0.02$ & $0.989\pm0.006$ & $0.621\pm0.081$ & & \\
	& FW-MRS-SVM & $0.833\pm0.015$ & $0.82\pm0.017$ & $0.752\pm0.022$ & $0.985\pm0.008$ & $0.559\pm0.09$ & & \\
	& Soft-MRS-Linear & $0.863\pm0.013$ & $0.825\pm0.013$ & $0.754\pm0.019$ & $0.989\pm0.005$ & $0.622\pm0.088$ & & \\
	& Soft-MRS-Exponential & $0.864\pm0.012$ & $0.825\pm0.012$ & $0.755\pm0.018$ & $0.989\pm0.006$ & $0.622\pm0.095$ & & \\


less_positive_class, 0.2
	& Uniform & $0.877\pm0.009$ & $0.848\pm0.01$ & $0.758\pm0.015$ & $0.989\pm0.00

In [7]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auprc_values = []
            std_auprc_values = []
            for dataset in datasets:
                mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                mean_auprc_values.append(np.round(mean_auprc, 3))

                std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                std_auprc_values.append(np.round(std_auprc, 3))

            print(f"\t& {method} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ & & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.828\pm0.017$ & $0.788\pm0.019$ & $0.456\pm0.035$ & $0.994\pm0.003$ & $0.803\pm0.05$ & & \\
	& PSA & $0.824\pm0.017$ & $0.78\pm0.022$ & $0.457\pm0.035$ & $0.994\pm0.003$ & $0.79\pm0.059$ & & \\
	& KMM & $0.808\pm0.022$ & $0.764\pm0.019$ & $0.454\pm0.031$ & $0.995\pm0.003$ & $0.779\pm0.052$ & & \\
	& MRS & $0.824\pm0.015$ & $0.787\pm0.021$ & $0.456\pm0.031$ & $0.995\pm0.002$ & $0.795\pm0.05$ & & \\
	& FW-MRS-RF & $0.816\pm0.017$ & $0.784\pm0.02$ & $0.458\pm0.035$ & $0.995\pm0.003$ & $0.78\pm0.053$ & & \\
	& FW-MRS-SVM & $0.776\pm0.023$ & $0.77\pm0.023$ & $0.455\pm0.038$ & $0.992\pm0.006$ & $0.742\pm0.059$ & & \\
	& Soft-MRS-Linear & $0.819\pm0.02$ & $0.773\pm0.019$ & $0.457\pm0.034$ & $0.995\pm0.003$ & $0.783\pm0.056$ & & \\
	& Soft-MRS-Exponential & $0.819\pm0.019$ & $0.772\pm0.019$ & $0.461\pm0.033$ & $0.995\pm0.003$ & $0.783\pm0.059$ & & \\


less_positive_class, 0.2
	& Uniform & $0.835\pm0.015$ & $0.804\pm0.014$ & $0.465\pm0.027$ & $0.995\pm0.